# 02 · Tool design and function schemas

Write tools a model can pick correctly and call safely: names and descriptions that make selection reliable, argument schemas that make wrong values impossible (`enum`, `required`, `strict`, `additionalProperties: false`), schemas generated from Python type hints, small observations, errors that teach, idempotent side effects, and the security rules for anything a model can trigger.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · The agent loop from scratch](01_agent_loop_from_scratch.ipynb)

## Why this matters in interviews

- "Same model, same prompt, one agent resolves 71% of tickets, the other 38%" (`ag06`): the usual answer is the tool surface, not the model.
- Tool definitions *are* prompts. Interviewers like candidates who fix a description before reaching for a bigger model.
- Security questions (never `eval`, sandboxing, approvals, idempotency) are where agent interviews turn into system-design interviews.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ag04` | What makes a good tool definition? |
| `ag06` | Two agents, same model, same tools, similar prompts. One resolves 71% of tickets, the other 38%. What explains it? |
| `ag07` | What is observation shaping and why does it matter so much? |
| `ag18` | How do you scope tools so the agent picks the right one? |
| `ag19` | How do you handle a tool that fails? |
| `ag25` | How do you handle irreversible actions? |
| `ag35` | What is the difference between structured output and tool calling? |
| `po18` | What is idempotency and why does an agent system need it more than a normal one? |
| `pr18` | How do you pass tool definitions efficiently? |
| `sf13` | How do you sandbox generated code? |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

import json
import random
import inspect
import hashlib
import subprocess
import tempfile
import os
from typing import Annotated, Literal, get_type_hints

import numpy as np
import matplotlib.pyplot as plt
import openai
import tiktoken
from pydantic import BaseModel, Field, TypeAdapter, ValidationError, create_model

random.seed(0)
rng = np.random.default_rng(0)
enc = tiktoken.get_encoding("o200k_base")          # the tokenizer of gpt-4o / gpt-4.1-family models


def n_tokens(obj) -> int:
    return len(enc.encode(obj if isinstance(obj, str) else json.dumps(obj)))

## 1. Names and descriptions decide which tool gets picked

**In plain English:** the model chooses a tool by reading its name and description, the way you would pick a function from its docstring. A vague description is a coin toss.

The experiment: eight real support requests, three toolsets that do *the same thing* but are described differently. We count how often the model calls exactly the right tool.

**Offline note:** the stand-in picks tools by word overlap between your request and the tool's name, description and parameter names. That makes this experiment *real* offline (it measures the text you wrote), and it is the same failure a real model shows in a milder form: it cannot pick well between tools it cannot tell apart.

In [ ]:
def tool(name, description, properties, required=None, strict=False):
    params = {"type": "object", "properties": properties,
              "required": list(properties) if required is None else required}
    if strict:
        params["additionalProperties"] = False
    spec = {"name": name, "description": description, "parameters": params}
    if strict:
        spec["strict"] = True
    return {"type": "function", "function": spec}


S = {"type": "string"}
VAGUE = [tool("get_data", "Gets data.", {"query": S}),
         tool("get_info", "Returns information.", {"query": S}),
         tool("lookup", "Looks things up.", {"query": S})]
NAMES_ONLY = [tool("get_order_status", "Gets an order.", {"order_id": S}),
              tool("search_help_articles", "Searches.", {"query": S}),
              tool("get_invoice", "Invoice tool.", {"customer_email": S})]
CLEAR = [tool("get_order_status", "Look up ONE order by its order id (like ORD-123456): status, shipping carrier, "
              "delivery date, tracking.", {"order_id": S}),
         tool("search_help_articles", "Search help-centre articles about refunds, returns, warranty, password reset "
              "and account settings. Use for 'how do I' and policy questions.", {"query": S}),
         tool("get_invoice", "Fetch a customer's invoice, bill, charges or payment receipt for a billing period.",
              {"customer_email": S})]

TESTS = [("Where is my order ORD-123456? Has it shipped?", "get_order_status"),
         ("When will order ORD-555001 be delivered?", "get_order_status"),
         ("Track my package for order ORD-777000", "get_order_status"),
         ("How do I reset my password?", "search_help_articles"),
         ("What is your refund policy for opened items?", "search_help_articles"),
         ("Does the warranty cover a cracked screen?", "search_help_articles"),
         ("Send me the invoice for March, my email is ana@example.com", "get_invoice"),
         ("Why was I charged twice on my bill? email bo@example.com", "get_invoice")]


def selection_accuracy(tools, show=False):
    right = 0
    for request, want in TESTS:
        msg = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": request}],
                                             tools=tools).choices[0].message
        got = [tc.function.name for tc in msg.tool_calls or []]
        right += got == [want]
        if show:
            print(f"  {'ok ' if got == [want] else 'BAD'} {request[:48]:48s} -> {got or 'no tool (answers from memory!)'}")
    return right / len(TESTS)


scores = {}
for label, tools in [("vague", VAGUE), ("names only", NAMES_ONLY), ("names + descriptions", CLEAR)]:
    print(label)
    scores[label] = selection_accuracy(tools, show=True)
print({k: f"{v:.0%}" for k, v in scores.items()})
assert scores["vague"] < scores["names only"] < scores["names + descriptions"] == 1.0

Two lessons from that output:

- With vague tools the model often calls **no tool at all** and answers from its own knowledge. That is worse than a wrong tool: the user gets a confident, made-up order status.
- Good names fix part of it; descriptions that list *what the tool returns* and *the words users actually say* ("bill", "charges", "delivered", "tracking") fix the rest.

What a good definition contains (`ag04`):

| Part | Bad | Good |
|---|---|---|
| name | `get_data` | `get_order_status` (verb_noun, one job) |
| description | "Gets an order." | what it returns, its limits, when to use a *different* tool, an example |
| parameters | free `query` string | typed fields, `enum` for closed sets, patterns (`^ORD-[0-9]{6}$`), ranges |
| errors | "400 Bad Request" | "status must be one of OPEN, PENDING, CLOSED; you sent 'open'" |

**Negative guidance** ("Do NOT use this to count orders; use `count_orders`") is very effective on real models. Our keyword stand-in cannot read "NOT", so it is not demonstrated here: a reminder that the stand-in shows the plumbing, not the intelligence.

## 2. Argument schemas: make wrong values impossible

A free-text `status` field invites "open", "Open", "opened", each a silent empty result. An `enum` makes the model choose from the legal values. **Strict mode** (`"strict": true`) goes further: the provider constrains decoding so the arguments *always* match the schema. It has two rules you must know:

- every property is listed in `required` (optional fields become `"type": ["string", "null"]`), and
- every object has `"additionalProperties": false`.

In [ ]:
search_orders = tool(
    "search_orders",
    "Find orders for ONE customer by email, optionally filtered by status. Returns at most 5, newest first.",
    {"customer_email": {"type": "string", "description": "the customer's email address"},
     "status": {"type": "string", "enum": ["OPEN", "PENDING", "SHIPPED", "CLOSED"]},
     "limit": {"type": ["integer", "null"], "description": "max results, default 5"}},
    strict=True)

msg = client.chat.completions.create(model=MODEL, tools=[search_orders], messages=[
    {"role": "user", "content": "Show my shipped orders, my email is ana@example.com"}]).choices[0].message
args = json.loads(msg.tool_calls[0].function.arguments)
print("arguments:", args)
assert args["status"] in ["OPEN", "PENDING", "SHIPPED", "CLOSED"]

broken = json.loads(json.dumps(search_orders))
del broken["function"]["parameters"]["additionalProperties"]
try:
    client.chat.completions.create(model=MODEL, tools=[broken], messages=[{"role": "user", "content": "hi"}])
except openai.BadRequestError as e:
    print("strict schema without additionalProperties -> 400:", e.body["message"][:110])

The stand-in rejects the broken strict schema exactly like the real API does: you find this out at the first request, not in code review.

**Structured output vs tool calling (`ag35`):** both constrain the model to JSON matching a schema. Structured output (`response_format=`) constrains the **final answer** and you are done. A tool call is an **intermediate action**: your code executes it and the loop continues. If the model must choose exactly one of a few actions and nothing loops, a structured output with a discriminated union is often simpler than a tool loop.

## 3. Generate schemas from Python functions

Hand-writing JSON Schema drifts out of sync with the code. Three ways to generate it:

In [ ]:
def find_orders(customer_email: Annotated[str, Field(description="the customer's email")],
                status: Literal["OPEN", "PENDING", "SHIPPED", "CLOSED"] = "OPEN",
                limit: Annotated[int, Field(ge=1, le=20)] = 5) -> list[dict]:
    """Find orders for one customer, newest first."""
    return []


# (a) pydantic reads the signature directly
print("(a) TypeAdapter:", json.dumps(TypeAdapter(find_orders).json_schema()))


# (b) a pydantic model -> a strict OpenAI tool, via the SDK's helper
class GetInvoice(BaseModel):
    """Fetch a customer's invoice for one month."""
    customer_email: str = Field(description="the customer's email")
    month: Literal["2026-07", "2026-08", "2026-09"]


print("(b) openai.pydantic_function_tool:", json.dumps(openai.pydantic_function_tool(GetInvoice))[:230], "...")

(c) is what LangChain's `@tool` and MCP's `@mcp.tool()` do under the hood: inspect the signature, build a pydantic model from the type hints, use the docstring as the description, and keep the model around to **validate the arguments the LLM sends**.

In [ ]:
REGISTRY = {}


def as_tool(risk="read"):
    """Decorator: register a function as a tool with a generated schema and an argument validator."""
    def register(fn):
        hints = get_type_hints(fn, include_extras=True)
        fields = {name: (hints.get(name, str), ... if p.default is p.empty else p.default)
                  for name, p in inspect.signature(fn).parameters.items()}
        Args = create_model(f"{fn.__name__}_args", **fields)
        schema = Args.model_json_schema()
        schema.pop("title", None)
        REGISTRY[fn.__name__] = {"fn": fn, "args": Args, "risk": risk, "schema": {
            "type": "function",
            "function": {"name": fn.__name__, "description": inspect.getdoc(fn), "parameters": schema}}}
        return fn
    return register


@as_tool()
def get_order_status(order_id: Annotated[str, Field(pattern=r"^ORD-\d{6}$", description="like ORD-123456")]) -> str:
    """Look up ONE order by its order id: status, carrier, delivery date."""
    return json.dumps({"order_id": order_id, "status": "SHIPPED", "carrier": "DHL", "eta": "2026-10-02"})


print(json.dumps(REGISTRY["get_order_status"]["schema"], indent=1))

The validator is the important half. The model's arguments are **untrusted input**: validate them before anything runs, and turn validation errors into sentences the model can fix.

In [ ]:
def humanize(err: ValidationError) -> str:
    """Pydantic errors -> one instructive sentence per problem."""
    parts = []
    for e in err.errors():
        field = ".".join(str(x) for x in e["loc"]) or "arguments"
        allowed = e.get("ctx", {}).get("expected") or e.get("ctx", {}).get("pattern")
        hint = f" (expected {allowed})" if allowed and str(allowed) not in e["msg"] else ""
        parts.append(f"{field}: {e['msg']}{hint}; you sent {e.get('input')!r}")
    return "INVALID ARGUMENTS - " + " | ".join(parts)


def execute(name: str, arguments: str) -> str:
    entry = REGISTRY.get(name)
    if entry is None:
        return f"ERROR: unknown tool {name!r}; available: {', '.join(REGISTRY)}"
    try:
        args = entry["args"].model_validate_json(arguments)
    except ValidationError as err:
        return humanize(err)
    return entry["fn"](**args.model_dump())


print(execute("get_order_status", '{"order_id": "ORD-123456"}'))
print(execute("get_order_status", '{"order_id": "123456"}'))
print(execute("get_order_status", '{"id": "ORD-123456"}'))

Compare what the model sees on a bad call:

| Observation | Next move for the model |
|---|---|
| `400 Bad Request` | guess, or give up |
| `Traceback (most recent call last): ...` | tokens spent teaching it nothing, and internals leaked |
| `order_id: String should match pattern '^ORD-\d{6}$'; you sent '123456'` | add the prefix and retry: fixed in one step |

Also fix the trivially fixable *inside the tool* instead of bouncing it back: accept "ord-123456" and upper-case it. Be strict about meaning, liberal about formatting.

## 4. Shape the observation: small, projected, paginated

**In plain English:** whatever a tool returns is pasted into the conversation and re-sent on every later step. A tool that dumps 50 raw records burns money and buries the answer (`ag07`).

In [ ]:
def raw_orders(n=50):
    statuses = ["OPEN", "PENDING", "SHIPPED", "CLOSED"]
    return [{"order_id": f"ORD-{100000 + i}", "created_at": f"2026-09-{1 + i % 28:02d}T10:{i % 60:02d}:00Z",
             "status": statuses[int(rng.integers(0, 4))], "total_cents": int(rng.integers(500, 30000)),
             "currency": "EUR", "customer": {"id": f"C-{9000 + i}", "email": "ana@example.com", "tier": "gold"},
             "shipping": {"carrier": "DHL", "tracking": f"JD{int(rng.integers(1e9, 9e9))}",
                          "address": {"street": "12 Rua Augusta", "city": "Lisbon", "zip": "1100-053"}},
             "items": [{"sku": f"SKU-{int(rng.integers(1000, 9999))}", "qty": 1, "price_cents": 1999}] * 2,
             "internal_flags": {"fraud_score": 0.02, "warehouse": "LIS-2", "sync_version": 17}}
            for i in range(n)]


def shape_orders(rows, fields=("order_id", "status", "total_cents", "created_at"), limit=5, offset=0):
    page = [{f: r[f] for f in fields} for r in rows[offset:offset + limit]]
    more = len(rows) - offset - len(page)
    note = f"showing {len(page)} of {len(rows)}; call again with offset={offset + limit} for more" if more > 0 else "end"
    return json.dumps({"orders": page, "note": note})


rows = raw_orders()
raw_tokens, shaped_tokens = n_tokens(rows), n_tokens(shape_orders(rows))
print(shape_orders(rows)[:230], "...")
print(f"\nraw: {raw_tokens} tokens | shaped: {shaped_tokens} tokens | {raw_tokens / shaped_tokens:.0f}x smaller")
assert shaped_tokens < 0.1 * raw_tokens

In [ ]:
steps = np.arange(1, 7)
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(steps, raw_tokens * steps, "o-", color="#c92a2a", label="raw payload stays in the history")
ax.plot(steps, shaped_tokens * steps, "o-", color="#2b8a3e", label="shaped observation")
ax.set_xlabel("agent step (the result is re-sent on every later call)")
ax.set_ylabel("tokens re-sent for this one result")
ax.set_title("One tool result, re-sent every step: shape it at the tool boundary")
ax.legend()
plt.show()

Observation-shaping moves (`ag07`): **project** to the fields the decision needs, **paginate** with an explicit continuation ("showing 5 of 50, call again with offset=5"), **store big payloads out of band** under an id the model can reference, **summarise** long text with a cheap model, and **normalise** formats across tools. The `internal_flags` field is also a small data leak: shaping is a security measure too.

## 5. Side effects: classify every tool, and make writes idempotent

| Risk class | Examples | Ceremony needed |
|---|---|---|
| read | look up an order, search articles | none |
| reversible write | add a note, draft an email | log it |
| irreversible / external | refund, send email, delete, change permissions | validation + limits in code + human approval + idempotency key |

**Idempotency (`po18`)**: agents retry, queues deliver twice, users double-click. A refund tool without an idempotency key will pay twice the first time a response times out *after* the money moved.

In [ ]:
class PaymentsAPI:
    """Fake payment provider. The first call applies the refund but the response is lost (a timeout)."""
    def __init__(self):
        self.refunds, self.by_key, self._drop_next_response = [], {}, True

    def refund(self, order_id, amount_cents, idempotency_key=None):
        if idempotency_key and idempotency_key in self.by_key:
            return self.by_key[idempotency_key]                  # replay: same result, no new money moves
        record = {"refund_id": f"RF-{len(self.refunds) + 1}", "order_id": order_id, "amount_cents": amount_cents}
        self.refunds.append(record)
        if idempotency_key:
            self.by_key[idempotency_key] = record
        if self._drop_next_response:
            self._drop_next_response = False
            raise TimeoutError("payment provider did not answer in 2 s")
        return record


def refund_with_retry(api, order_id, amount_cents, use_key, attempts=2):
    key = hashlib.sha256(f"refund:{order_id}:{amount_cents}".encode()).hexdigest()[:16] if use_key else None
    for attempt in range(attempts):
        try:
            return api.refund(order_id, amount_cents, idempotency_key=key)
        except TimeoutError:
            continue                                             # the harness retries, as it should
    raise RuntimeError("refund failed")


for use_key in (False, True):
    api = PaymentsAPI()
    refund_with_retry(api, "ORD-448812", 8920, use_key)
    print(f"idempotency key={use_key!s:5}: refunds actually issued = {len(api.refunds)}")
    assert len(api.refunds) == (1 if use_key else 2)

The key is **deterministic**: derived from the logical action (order id + amount), so every retry, from any process, carries the same key. A random UUID per attempt would defeat the purpose.

## 6. Security: the model's tool call is a request from a stranger

Four rules, each with a demonstration:

1. **Never `eval` or `exec` model output.**
2. **Allowlist and validate**: only registered tools, only validated arguments, paths resolved inside an allowed root.
3. **Sandbox generated code**: separate process, time limit, no secrets, thrown away afterwards (`sf13`).
4. **Gate destructive tools**: they do not run without an explicit approval (full approval gates in notebook [06](06_guardrails_hitl_and_failure_modes.ipynb)).

In [ ]:
# 1. eval runs ANY code. This payload is harmless, but it proves arbitrary code execution:
payload = "__import__('os').getcwd()"
print("eval(payload) ran code and returned:", eval(payload))

# 2. Allowlisted file access: resolve the path, then check it stays under the root
root = Path(tempfile.mkdtemp())
(root / "docs").mkdir()
(root / "docs" / "faq.txt").write_text("Refunds take 14 days.")
(root / "secret.env").write_text("API_KEY=sk-not-for-the-model")
DOCS = (root / "docs").resolve()


def read_doc(path: str) -> str:
    target = (DOCS / path).resolve()
    if not target.is_relative_to(DOCS):
        return f"REFUSED: {path!r} is outside the documents folder"
    return target.read_text() if target.exists() else f"ERROR: no document {path!r}"


print("read_doc('faq.txt')        ->", read_doc("faq.txt"))
print("read_doc('../secret.env')  ->", read_doc("../secret.env"))
assert read_doc("../secret.env").startswith("REFUSED")

In [ ]:
# 3. Run generated code in a separate, time-limited process with an empty working dir and no secrets in env
def run_untrusted(code: str, timeout_s: float = 1.0) -> str:
    env = {k: os.environ[k] for k in ("SYSTEMROOT", "PATH") if k in os.environ}   # no API keys inherited
    try:
        r = subprocess.run([sys.executable, "-I", "-c", code], cwd=tempfile.mkdtemp(), env=env,
                           capture_output=True, text=True, timeout=timeout_s)
        return (r.stdout or r.stderr).strip()[:200]
    except subprocess.TimeoutExpired:
        return f"KILLED: exceeded {timeout_s} s"


looped = run_untrusted("while True: pass")
print("sum of squares ->", run_untrusted("print(sum(i*i for i in range(10)))"))
print("infinite loop  ->", looped)
print("read my env    ->", run_untrusted("import os; print(sorted(os.environ))"))
assert looped.startswith("KILLED")

That subprocess gives a time limit and no inherited secrets, but it still runs as *you*, with your files and your network. A real sandbox adds isolation: a container or microVM (Docker with no network and a read-only filesystem, gVisor, Firecracker, or a hosted code sandbox), CPU and memory limits, and a fresh environment per run.

In [ ]:
# 4. Destructive tools need an approval token that only a human flow can create
@as_tool(risk="destructive")
def delete_customer(customer_id: Annotated[str, Field(pattern=r"^C-\d{4}$")]) -> str:
    """Permanently delete a customer and all their data. Irreversible."""
    return f"deleted {customer_id}"


def execute_guarded(name, arguments, approvals=frozenset()):
    entry = REGISTRY.get(name)
    if entry and entry["risk"] == "destructive" and (name, arguments) not in approvals:
        return f"APPROVAL_REQUIRED: {name}({arguments}) is irreversible; a human must approve this exact call."
    return execute(name, arguments)


call = ("delete_customer", '{"customer_id": "C-9001"}')
print(execute_guarded(*call))
print(execute_guarded(*call, approvals={call}))
assert execute_guarded(*call).startswith("APPROVAL_REQUIRED")

The approval is bound to the **exact** call (name + arguments), so an approval for `C-9001` cannot be replayed to delete `C-9002`. Limits (maximum refund, calls per day) belong in code like this, never only in the prompt: a prompt is advice, code is a rule.

## 7. How many tools? Accuracy and cost both get worse

Every tool definition is sent on every call (`pr18`), and every extra tool is another option to confuse with the right one. Below we add realistic near-duplicates (a destructive `cancel_order` next to `get_order_status`, `get_refund_status` next to the refund *policy*, `list_invoices` next to `get_invoice`, ...) and re-run the eight requests.

In [ ]:
DISTRACTORS = [
    tool("cancel_order", "Cancel an order that has not shipped yet.", {"order_id": S}),
    tool("get_order_history", "List all past orders for a customer email: order ids, dates, totals.", {"customer_email": S}),
    tool("get_refund_status", "Check the status of a refund for an order.", {"order_id": S}),
    tool("update_delivery_address", "Change the delivery address of an order before it ships.", {"order_id": S, "address": S}),
    tool("file_warranty_claim", "Open a warranty claim for a damaged or cracked product.", {"order_id": S}),
    tool("list_invoices", "List every invoice and bill for a customer email.", {"customer_email": S}),
    tool("track_parcel", "Track a parcel with the carrier tracking number.", {"tracking_number": S}),
    tool("get_shipping_rates", "Quote shipping rates and delivery times for a new order.", {"destination": S}),
    tool("get_payment_methods", "List saved cards and payment methods for a customer.", {"customer_email": S}),
    tool("search_products", "Search the product catalogue by name or category.", {"query": S}),
    tool("create_support_ticket", "Open a ticket for a human agent.", {"description": S}),
    tool("get_account_settings", "Read the user's account settings and preferences.", {"customer_email": S}),
]
sizes, accs, costs = [], [], []
for extra in (0, 3, 6, 9, 12):
    tools = CLEAR + DISTRACTORS[:extra]
    sizes.append(len(tools)); accs.append(selection_accuracy(tools)); costs.append(n_tokens(tools))
    print(f"{len(tools):2d} tools: exactly-right-tool rate {accs[-1]:.0%}, definitions = {costs[-1]} tokens per call")
assert accs[0] == 1.0 and accs[-1] < accs[0] and costs[-1] > 3 * costs[0]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(sizes, np.array(accs) * 100, "o-", color="#4c6ef5", label="exactly the right tool (%)")
ax.set_xlabel("number of tools offered"); ax.set_ylabel("selection accuracy (%)")
ax.set_ylim(-5, 105)
ax2 = ax.twinx()
ax2.bar(sizes, costs, width=0.8, alpha=0.25, color="#fab005", label="definition tokens per call")
ax2.set_ylabel("tokens of tool definitions")
ax.set_title("More (and overlapping) tools: worse selection, more tokens on every call")
ax.legend(loc="lower left", fontsize=8); ax2.legend(loc="upper center", fontsize=8)
plt.show()

Read this chart honestly. The stand-in is a keyword matcher, so it is confused more easily than a real model and its accuracy falls faster. The *direction* is real: selection errors grow with the number of similar tools (OpenAI's function-calling guide suggests keeping fewer than about 20 functions available at a time), and the token cost is exact: each definition is paid for on every single call. Notice the worst failure: "Where is my order?" also triggering `cancel_order`.

The fixes (`ag18`):

- **Phase scoping**: expose only the tools legal at this point of the task. Below, the "answering questions" phase gets three tools and the account-changing tools appear only in a separate "actions" phase.
- **Merge near-duplicates**: one `get_invoices(customer_email, month=None)` instead of `get_invoice` + `list_invoices`; one `search(source=enum)` instead of five search tools.
- **Tool retrieval** when you truly have hundreds: embed the descriptions and offer the top-k per request (it needs semantic embeddings; our offline hashed embeddings are keyword-level, so it is not demonstrated here).
- Keep the tool list in a **fixed order**: definitions are part of the prompt prefix, and a shuffled list breaks provider prompt caching.

In [ ]:
PHASES = {"answer_questions": CLEAR,
          "account_actions": [t for t in DISTRACTORS if t["function"]["name"] in
                              ("cancel_order", "update_delivery_address", "file_warranty_claim")]}
scoped = selection_accuracy(PHASES["answer_questions"])
print(f"all {len(CLEAR + DISTRACTORS)} tools: {accs[-1]:.0%} | phase-scoped ({len(PHASES['answer_questions'])} tools): {scoped:.0%}")
assert scoped == 1.0

## 8. The tool-design checklist

| # | Check | Why |
|---|---|---|
| 1 | `verb_noun` name, one job per tool | selection starts from the name |
| 2 | description says what it returns, its limits, when to use another tool, and the words users say | descriptions are prompts |
| 3 | `enum` for closed sets, patterns and ranges in the schema, accurate `required` | makes wrong values impossible |
| 4 | `strict: true` + `additionalProperties: false` where supported | arguments always parse |
| 5 | schema generated from the code (type hints / pydantic), validated on every call | no drift, untrusted input checked |
| 6 | small observations: projected, paginated, big payloads by reference | cost and attention |
| 7 | errors name the problem and the legal values; retryable vs terminal is explicit | recovery in one step |
| 8 | each tool classified read / write / irreversible; irreversible ones gated and limited in code | blast radius |
| 9 | idempotency key on every write, derived from the logical action | retries do not double-charge |
| 10 | no `eval`, allowlisted paths, sandboxed code, no secrets in the tool's environment | the caller is untrusted |
| 11 | only the tools needed for this phase, in a stable order | accuracy, tokens, caching |

## Try it yourself

**Exercise 1.** The tool `inv(q)` below is useless to the model. Rewrite its name, description and parameter name (same function) so that all three stock requests select it.

In [ ]:
# Solution — try it yourself first, then run this
bad = tool("inv", "Inventory endpoint.", {"q": S})
good = tool("check_stock", "Check whether a product is in stock and how many units are available, by product SKU "
            "(like SKU-1234). Use for availability and 'do you have' questions.", {"sku": S})
stock_requests = ["Is SKU-1234 in stock?", "Do you have SKU-9876 available?", "How many units of SKU-5555 are left?"]
picked = {}
for label, t in [("bad", bad), ("good", good)]:
    picked[label] = sum(bool(client.chat.completions.create(model=MODEL, tools=[t], messages=[
        {"role": "user", "content": r}]).choices[0].message.tool_calls) for r in stock_requests)
    print(f"{label}: selected on {picked[label]}/3 requests")
assert picked["good"] == 3 and picked["bad"] < 3

**Exercise 2.** Write `shape_articles(articles, max_chars=120, limit=3)` that trims each article body to `max_chars` (ending with "…") and adds a "showing N of M" note. Check the shaped output is under 25% of the raw tokens.

In [ ]:
# Solution — try it yourself first, then run this
articles = [{"id": f"KB-{i}", "title": f"How refunds work, part {i}",
             "body": "Refunds are issued within 14 days of purchase for unused items. " * 12} for i in range(8)]


def shape_articles(articles, max_chars=120, limit=3):
    page = [{"id": a["id"], "title": a["title"],
             "body": a["body"][:max_chars].rstrip() + ("…" if len(a["body"]) > max_chars else "")}
            for a in articles[:limit]]
    return json.dumps({"articles": page, "note": f"showing {len(page)} of {len(articles)}"})


print(shape_articles(articles)[:200], "...")
ratio = n_tokens(shape_articles(articles)) / n_tokens(articles)
print(f"shaped / raw tokens = {ratio:.0%}")
assert ratio < 0.25

**Exercise 3.** Make a `send_email(to, subject, body)` tool idempotent: a second call with the same logical email (same recipient, subject and body) must not send again. Also refuse any recipient outside `@example.com` (an egress allowlist).

In [ ]:
# Solution — try it yourself first, then run this
SENT = {}


def send_email(to: str, subject: str, body: str) -> str:
    if not to.endswith("@example.com"):
        return f"REFUSED: {to!r} is not an allowed recipient domain"
    key = hashlib.sha256(f"{to}|{subject}|{body}".encode()).hexdigest()[:12]
    if key in SENT:
        return f"already sent (message {SENT[key]}); not sending again"
    SENT[key] = f"MSG-{len(SENT) + 1}"
    return f"sent {SENT[key]} to {to}"


print(send_email("ana@example.com", "Your refund", "Refund RF-1 issued."))
print(send_email("ana@example.com", "Your refund", "Refund RF-1 issued."))
print(send_email("attacker@evil.test", "data", "all customer emails"))
assert len(SENT) == 1

## Say it in an interview

**30-second answer (`ag04`).** "A good tool has a verb-noun name with one job, a description that says what it returns, its limits and when to use a different tool, and a schema that makes wrong values impossible: enums, patterns, ranges, accurate required fields, strict mode with additionalProperties false. I generate the schema from the function's type hints and validate every call against it, because the model's arguments are untrusted input. Observations are projected and paginated. Errors name the mistake and the legal values. Writes are classified by risk, gated when irreversible, and carry a deterministic idempotency key."

**`ag06`, the 71% vs 38% question.** None of the differences is the prompt: tool descriptions, enums vs free strings, instructive vs stack-trace errors, observation size, history policy and stop conditions. Say that explicitly.

**Numbers.** In this notebook one shaped observation was many times smaller than the raw payload, and 15 tool definitions cost several times the tokens of 3 on every call. Keep the active tool list small (OpenAI suggests fewer than ~20).

**Traps.**
- "The model will only call tools correctly" — no: validate every argument, enforce limits in code.
- Random idempotency keys per retry (useless) instead of keys derived from the action.
- `eval` in a calculator, `open(path)` without resolving and checking the root.
- Giving an agent a destructive tool it does not need (excessive agency, `sf15`).
- Shuffling the tool list per request (breaks prompt caching).

## Next

- [03 · ReAct and plan-and-execute](03_react_and_plan_and_execute.ipynb) · [06 · Guardrails, HITL and failure modes](06_guardrails_hitl_and_failure_modes.ipynb) · [07 · MCP](07_mcp_model_context_protocol.ipynb) (tools served over a protocol).
- [Structured outputs with pydantic](../08_llm_apps/03_structured_outputs_with_pydantic.ipynb) · [LangChain structured output and tools](../13_langchain/03_structured_output_and_tools.ipynb) · [Guardrails, PII and prompt injection](../16_production/07_guardrails_pii_and_prompt_injection.ipynb).
- Theory: [agentic_ai_flow course](../../agentic_ai_flow/index.html) (chapters *Tools & function calling*, *Harness engineering*) and the [interview bank](../../ai_interview_prep/index.html).